# CloudCommerce Policy RAG

End-to-end notebook for testing the policy retrieval and answer pipeline.

**Flow**

1. Load Markdown policies with `TextLoader`
2. Split by Markdown headers and chunk size
3. Index chunks in persistent Chroma
4. Retrieve relevant policy sections
5. Answer through the official retrieval chain
6. Fall back to `I don't know` when no policy matches

**Requirements**

- The first code cell locates the project root automatically
- `.env` must contain your provider and API key settings
- Restart the kernel after changing `.env`

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find the project root (pyproject.toml).")
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/arun/Work/WorkCloudSathi/cloudcommerce-order-support-agent


In [2]:
from collections import Counter

from app.config import get_settings
from app.rag import (
    UNKNOWN_ANSWER,
    answer_policy_question,
    build_policy_chain,
    build_policy_retriever,
    build_policy_store,
    citation_for,
    format_citations,
    load_policy_documents,
)

## 1. Configuration

In [3]:
settings = get_settings()

print("Policy docs:", settings.policy_docs_path)
print("Chroma path:", settings.chroma_persist_path)
print("Collection:", settings.chroma_collection_name)
print("Chunk size:", settings.rag_chunk_size)
print("Chunk overlap:", settings.rag_chunk_overlap)
print("Top-k:", settings.rag_top_k)
print("Min score:", settings.rag_min_score)

Policy docs: data/policies
Chroma path: data/chroma
Collection: policies
Chunk size: 700
Chunk overlap: 120
Top-k: 4
Min score: 0.35


## 2. Load and split policies

`TextLoader` reads each Markdown file. `MarkdownHeaderTextSplitter` keeps the
`title` and `section` metadata, and `RecursiveCharacterTextSplitter` controls
chunk size.

In [4]:
documents = load_policy_documents(
    settings.policy_docs_path,
    chunk_size=settings.rag_chunk_size,
    chunk_overlap=settings.rag_chunk_overlap,
)

print(f"Loaded {len(documents)} chunks")
print(Counter(document.metadata["source"] for document in documents))
print()
print(documents[0].metadata)
print()
print(documents[0].page_content[:300])

Loaded 17 chunks
Counter({'returns.md': 5, 'cancellations.md': 4, 'shipping.md': 4, 'warranty.md': 4})

{'title': 'Cancellations Policy', 'section': 'When cancellation is allowed', 'source': 'cancellations.md'}

# Cancellations Policy  
## When cancellation is allowed  
Orders can only be cancelled while the status is "processing". Support must check the current order status before confirming a cancellation.  
Orders that are shipped, delivered, cancelled, or refunded cannot be cancelled. For shipped orders


## 3. Build the Chroma index

In [5]:
store = build_policy_store(settings)
print(f"Indexed chunks: {len(store.get()['ids'])}")

/home/arun/Work/WorkCloudSathi/cloudcommerce-order-support-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9644.23it/s]


Indexed chunks: 17


## 4. Create the retriever

In [6]:
retriever = build_policy_retriever(settings)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x77cab8d80fb0>, search_type='similarity_score_threshold', search_kwargs={'k': 4, 'score_threshold': 0.35})

## 5. Retrieve policy chunks

In [7]:
questions = [
    "Can I return an item after 14 days?",
    "Can I cancel an order that is still processing?",
    "How long do refunds take?",
]

for question in questions:
    matches = retriever.invoke(question)
    print(f"\nQ: {question}  ({len(matches)} matches)")
    for document in matches:
        print(f"- {citation_for(document)}")


Q: Can I return an item after 14 days?  (4 matches)
- returns.md › Return window
- warranty.md › Warranty and returns
- returns.md › Non-returnable items
- shipping.md › Lost or delayed packages

Q: Can I cancel an order that is still processing?  (4 matches)
- cancellations.md › When cancellation is allowed
- cancellations.md › Personalized and made-to-order items
- cancellations.md › How support should handle a cancellation
- cancellations.md › Refund timing

Q: How long do refunds take?  (4 matches)
- returns.md › Refund timing
- cancellations.md › Refund timing
- returns.md › Return window
- shipping.md › Processing and delivery


### Relevance scores

In [8]:
for question in questions:
    results = store.similarity_search_with_relevance_scores(
        question,
        k=settings.rag_top_k,
    )
    print(f"\nQ: {question}")
    for document, score in results:
        print(f"  {score:.3f}  {citation_for(document)}")


Q: Can I return an item after 14 days?
  0.704  returns.md › Return window
  0.662  warranty.md › Warranty and returns
  0.470  returns.md › Non-returnable items
  0.466  shipping.md › Lost or delayed packages

Q: Can I cancel an order that is still processing?
  0.758  cancellations.md › When cancellation is allowed
  0.657  cancellations.md › Personalized and made-to-order items
  0.631  cancellations.md › How support should handle a cancellation
  0.458  cancellations.md › Refund timing

Q: How long do refunds take?
  0.817  returns.md › Refund timing
  0.761  cancellations.md › Refund timing
  0.562  returns.md › Return window
  0.394  shipping.md › Processing and delivery


## 6. Build the RAG chain

In [9]:
chain = build_policy_chain(settings)
response = chain.invoke({"input": "How long do refunds take?"})

print(response["answer"])
print()
print(format_citations(response["context"]))

Refunds are processed to the original payment method and typically appear within 5–7 business days after the refund is processed.

- returns.md › Refund timing
- cancellations.md › Refund timing
- returns.md › Return window
- shipping.md › Processing and delivery


## 7. Ask a question with citations

In [10]:
def ask(question: str) -> None:
    response = answer_policy_question(question)
    print(f"Q: {question}")
    print(f"A: {response['answer']}")

    citations = format_citations(response.get("context", []))
    if citations:
        print(citations)


ask("Can I cancel an order that is still processing?")

Q: Can I cancel an order that is still processing?
A: Yes—orders can be cancelled while their status is **“processing.”** Support will first verify that the order belongs to you and that it’s still in that status. If it’s a personalized or made‑to‑order item and production has already begun, cancellation isn’t possible, and you’d need to follow the return or warranty process instead.
- cancellations.md › When cancellation is allowed
- cancellations.md › Personalized and made-to-order items
- cancellations.md › How support should handle a cancellation
- cancellations.md › Refund timing


## 8. Uncovered question

No policy chunks pass the score threshold, so the pipeline returns a
deterministic fallback instead of calling the model.

In [11]:
response = answer_policy_question("Do you offer student discounts?")
print(response["answer"])
print("Matches:", len(response["context"]))
assert response["answer"] == UNKNOWN_ANSWER

No relevant docs were retrieved using the relevance score threshold 0.35


I don't know based on the available policies.
Matches: 0


## 9. Rebuild the index (optional)

Uncomment and run this cell after changing policy files or chunk settings.
Restart the kernel afterwards so cached settings and retrievers are refreshed.

In [ ]:
# import shutil
# from app.rag import get_policy_chain, get_policy_retriever
#
# shutil.rmtree(settings.chroma_persist_path, ignore_errors=True)
# get_policy_retriever.cache_clear()
# get_policy_chain.cache_clear()
#
# print("Index removed. Re-run the index and chain cells.")